# The Battle Class
This class is for making a Battle object.  A battle object takes a log (in the form of a string) as input.  From that string, it will gather the information of usernames, pre-battle ratings, post-battle ratings (if the match was rated), pokemon on each team, battle start and end time, and winner.  It has a nice string representation.

Next goal: allow it to take a .json file(name) as input instead of the raw log?

In [80]:
class Battle:
    def __init__(self,log):
        self.player_names = ["",""]
        self.old_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings at the start of the match
        self.new_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings after the end of the match
        self.winner = ""
        self.loser = ""
        self.teams = {} # a dictionary where the keys are player usernames and the values are sets containing the pokemon names
        self.time_list = [] # for keeping track of all move times, gets thrown away now.  Could be used to check for time-out forfeits?
        self.start_time = 0
        self.end_time = 0
        self.rated = False
        self.log_parser(log)

    def __repr__(self):
        to_return = f"This was a battle between {self.player_names[0]} (pre-match rating {self.old_player_ratings[self.player_names[0]]}) and {self.player_names[1]} (pre-match rating {self.old_player_ratings[self.player_names[1]]}).\n"
        for index in range(2):
            to_return += f"{self.player_names[index]}'s team was {self.teams[self.player_names[index]]}.\n"
        to_return += f"{self.winner} won!\n"
        if self.rated:
            to_return += f"{self.winner}'s rating increased to {self.new_player_ratings[self.winner]}.\n"
            to_return += f"{self.loser}'s rating fell to {self.new_player_ratings[self.loser]}."
        else:
            to_return += "This was an unrated match, so no one's rating changed."
        return to_return

    def log_parser(self,log):
        for i in range(len(log)):
            # generates player usernames, initializes the teams dict with an empty set for each player, sets the old_player_ratings
            if log.startswith("|player|",i):
                j = i+11
                # grabs the index of the player: 1 if p2, 0 else; this way, we avoid the assumption that player 1's name is printed first
                player_index = int(log.startswith("|p2|",i+7))
                while log[j] != "|":
                    self.player_names[player_index] += log[j]
                    j += 1
                # initializes the teams as empty sets
                self.teams[self.player_names[player_index]] = set()
                # grabs the initial ratings of each player; tricky because you want to grab the last 3 or 4 characters of the line depending on 
                # whether the player's rating is 3 or 4 digits
                while log[j] != "\n":
                    j += 1
                # j is now the index of the \n character at the end of the line
                k = 0
                while log[j - k] != "|":
                    k = k+1
                # j - k is now the index of the | before the player's rating
                self.old_player_ratings[self.player_names[player_index]] = int(log[j-k+1:j])
            # generates time list
            if log.startswith("|t:",i):
                j = i + 4
                time_string = ""
                while log[j] != "\n":
                    time_string += log[j]
                    j += 1
                self.time_list.append(int(time_string))
            # generate team list
            if log.startswith("|switch|",i):
                pokemon_name = ""
                j = i+13
                while log[j] != "|":
                    pokemon_name += log[j]
                    j += 1
                if log.startswith("p1a",i+8):
                    player_name = self.player_names[0]
                else:
                    player_name = self.player_names[1]
                self.teams[player_name].add(pokemon_name)
            # identifies winner and loser
            if log.startswith("|win|",i):
                j = i + 5
                while log[j] != "\n":
                    self.winner += log[j]
                    j += 1
                winner_index = self.player_names.index(self.winner)
                self.loser = self.player_names[1-winner_index]
            if log.startswith(" for winning)",i):
                delta_rating = int(log[i-2:i]) # assumes that the change in rating will be two digits
                self.new_player_ratings[self.winner] = self.old_player_ratings[self.winner] + delta_rating
                self.new_player_ratings[self.loser] = self.old_player_ratings[self.loser] - delta_rating
                self.rated = True
        self.start_time = self.time_list[0]
        self.end_time = self.time_list[-1]

## Examples
Here, you can see how the Battle class functions on a couple of logs that I grabbed from Showdown on June 15.  It's pretty straightforward

In [81]:
log1 = "|j|☆xValk_\n|j|☆manwhocouldntcook\n|t:|1781540286\n|gametype|singles\n|player|p1|xValk_|ltsurge|2138\n|player|p2|manwhocouldntcook|valerie|2146\n|gen|9\n|tier|[Gen 9] Random Battle\n|rated|\n|rule|Species Clause: Limit one of each Pokémon\n|rule|HP Percentage Mod: HP is shown in percentages\n|rule|Sleep Clause Mod: Limit one foe put to sleep\n|rule|Illusion Level Mod: Illusion disguises the Pokémon's true level\n|\n|t:|1781540286\n|teamsize|p1|6\n|teamsize|p2|6\n|start\n|switch|p1a: Ho-Oh|Ho-Oh, L71|268/268\n|switch|p2a: Scrafty|Scrafty, L83, M|244/244\n|turn|1\n|\n|t:|1781540296\n|switch|p2a: Incineroar|Incineroar, L82, F|290/290\n|-ability|p2a: Incineroar|Intimidate|boost\n|-unboost|p1a: Ho-Oh|atk|1\n|move|p1a: Ho-Oh|Brave Bird|p2a: Incineroar\n|-damage|p2a: Incineroar|206/290\n|-damage|p1a: Ho-Oh|240/268|[from] Recoil\n|\n|upkeep\n|turn|2\n|\n|t:|1781540301\n|switch|p1a: Carbink|Carbink, L90|236/236\n|move|p2a: Incineroar|Knock Off|p1a: Carbink\n|-resisted|p1a: Carbink\n|-damage|p1a: Carbink|205/236\n|-enditem|p1a: Carbink|Chesto Berry|[from] move: Knock Off|[of] p2a: Incineroar\n|\n|upkeep\n|turn|3\n|inactive|Battle timer is ON: inactive players will automatically lose when time's up. (requested by manwhocouldntcook)\n|\n|t:|1781540308\n|switch|p2a: Vileplume|Vileplume, L85, F|266/266\n|move|p1a: Carbink|Moonblast|p2a: Vileplume\n|-resisted|p2a: Vileplume\n|-damage|p2a: Vileplume|230/266\n|\n|-heal|p2a: Vileplume|246/266|[from] item: Leftovers\n|upkeep\n|turn|4\n|\n|t:|1781540313\n|switch|p1a: Victreebel|Victreebel, L90, M|290/290\n|move|p2a: Vileplume|Sleep Powder|p1a: Victreebel\n|-immune|p1a: Victreebel\n|\n|-heal|p2a: Vileplume|262/266|[from] item: Leftovers\n|upkeep\n|turn|5\n|\n|t:|1781540316\n|move|p1a: Victreebel|Sunny Day|p1a: Victreebel\n|-weather|SunnyDay\n|move|p2a: Vileplume|Sludge Bomb|p1a: Victreebel\n|-damage|p1a: Victreebel|175/290\n|\n|-weather|SunnyDay|[upkeep]\n|-heal|p2a: Vileplume|266/266|[from] item: Leftovers\n|upkeep\n|turn|6\n|\n|t:|1781540320\n|switch|p2a: Incineroar|Incineroar, L82, F|206/290\n|-ability|p2a: Incineroar|Intimidate|boost\n|-unboost|p1a: Victreebel|atk|1\n|move|p1a: Victreebel|Sludge Wave|p2a: Incineroar\n|-damage|p2a: Incineroar|62/290\n|-damage|p1a: Victreebel|146/290|[from] item: Life Orb\n|\n|-weather|SunnyDay|[upkeep]\n|upkeep\n|turn|7\n|\n|t:|1781540325\n|move|p1a: Victreebel|Sludge Wave|p2a: Incineroar\n|-damage|p2a: Incineroar|0 fnt\n|faint|p2a: Incineroar\n|-damage|p1a: Victreebel|117/290|[from] item: Life Orb\n|\n|-weather|SunnyDay|[upkeep]\n|upkeep\n|\n|t:|1781540335\n|switch|p2a: Scrafty|Scrafty, L83, M|244/244\n|turn|8\n|\n|t:|1781540338\n|move|p1a: Victreebel|Weather Ball|p2a: Scrafty\n|-damage|p2a: Scrafty|121/244\n|-damage|p1a: Victreebel|88/290|[from] item: Life Orb\n|move|p2a: Scrafty|Knock Off|p1a: Victreebel\n|-damage|p1a: Victreebel|0 fnt\n|-enditem|p1a: Victreebel|Life Orb|[from] move: Knock Off|[of] p2a: Scrafty\n|faint|p1a: Victreebel\n|\n|-weather|SunnyDay|[upkeep]\n|-heal|p2a: Scrafty|136/244|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540342\n|switch|p1a: Ho-Oh|Ho-Oh, L71|268/268\n|turn|9\n|\n|t:|1781540347\n|move|p1a: Ho-Oh|Sacred Fire|p2a: Scrafty\n|-damage|p2a: Scrafty|24/244\n|-status|p2a: Scrafty|brn\n|move|p2a: Scrafty|Rest|p2a: Scrafty\n|-status|p2a: Scrafty|slp|[from] move: Rest\n|-heal|p2a: Scrafty|244/244 slp|[silent]\n|\n|-weather|none\n|upkeep\n|turn|10\n|\n|t:|1781540353\n|switch|p2a: Jolteon|Jolteon, L84, F|246/246\n|move|p1a: Ho-Oh|Brave Bird|p2a: Jolteon\n|-resisted|p2a: Jolteon\n|-damage|p2a: Jolteon|169/246\n|-damage|p1a: Ho-Oh|243/268|[from] Recoil\n|\n|-heal|p2a: Jolteon|184/246|[from] item: Leftovers\n|upkeep\n|turn|11\n|\n|t:|1781540357\n|switch|p1a: Carbink|Carbink, L90|205/236\n|move|p2a: Jolteon|Substitute|p2a: Jolteon\n|-start|p2a: Jolteon|Substitute\n|-damage|p2a: Jolteon|123/246\n|\n|-heal|p2a: Jolteon|138/246|[from] item: Leftovers\n|upkeep\n|turn|12\n|\n|t:|1781540362\n|move|p2a: Jolteon|Thunderbolt|p1a: Carbink\n|-damage|p1a: Carbink|141/236\n|move|p1a: Carbink|Body Press|p2a: Jolteon\n|-end|p2a: Jolteon|Substitute\n|\n|-heal|p2a: Jolteon|153/246|[from] item: Leftovers\n|upkeep\n|turn|13\n|\n|t:|1781540368\n|switch|p2a: Vileplume|Vileplume, L85, F|266/266\n|switch|p1a: Ho-Oh|Ho-Oh, L71|268/268\n|\n|upkeep\n|turn|14\n|\n|t:|1781540372\n|move|p1a: Ho-Oh|Sacred Fire|p2a: Vileplume\n|-supereffective|p2a: Vileplume\n|-damage|p2a: Vileplume|68/266\n|-status|p2a: Vileplume|brn\n|move|p2a: Vileplume|Sleep Powder|p1a: Ho-Oh\n|-status|p1a: Ho-Oh|slp|[from] move: Sleep Powder\n|\n|-heal|p2a: Vileplume|84/266 brn|[from] item: Leftovers\n|-damage|p2a: Vileplume|68/266 brn|[from] brn\n|upkeep\n|turn|15\n|\n|t:|1781540377\n|cant|p1a: Ho-Oh|slp\n|move|p2a: Vileplume|Strength Sap|p1a: Ho-Oh\n|-unboost|p1a: Ho-Oh|atk|1\n|-heal|p2a: Vileplume|266/266 brn\n|\n|-damage|p2a: Vileplume|250/266 brn|[from] brn\n|upkeep\n|turn|16\n|\n|t:|1781540380\n|switch|p2a: Scrafty|Scrafty, L83, M|244/244 slp\n|-curestatus|p1a: Ho-Oh|slp|[msg]\n|move|p1a: Ho-Oh|Brave Bird|p2a: Scrafty\n|-supereffective|p2a: Scrafty\n|-damage|p2a: Scrafty|112/244 slp\n|-damage|p1a: Ho-Oh|224/268|[from] Recoil\n|\n|-heal|p2a: Scrafty|127/244 slp|[from] item: Leftovers\n|upkeep\n|turn|17\n|\n|t:|1781540387\n|move|p1a: Ho-Oh|Earthquake|p2a: Scrafty\n|-damage|p2a: Scrafty|92/244 slp\n|cant|p2a: Scrafty|slp\n|\n|-heal|p2a: Scrafty|107/244 slp|[from] item: Leftovers\n|upkeep\n|turn|18\n|\n|t:|1781540390\n|switch|p2a: Vileplume|Vileplume, L85, F|250/266 brn\n|move|p1a: Ho-Oh|Brave Bird|p2a: Vileplume\n|-supereffective|p2a: Vileplume\n|-damage|p2a: Vileplume|88/266 brn\n|-damage|p1a: Ho-Oh|171/268|[from] Recoil\n|\n|-heal|p2a: Vileplume|104/266 brn|[from] item: Leftovers\n|-damage|p2a: Vileplume|88/266 brn|[from] brn\n|upkeep\n|turn|19\n|\n|t:|1781540394\n|move|p1a: Ho-Oh|Sacred Fire|p2a: Vileplume\n|-supereffective|p2a: Vileplume\n|-damage|p2a: Vileplume|0 fnt\n|faint|p2a: Vileplume\n|\n|upkeep\n|\n|t:|1781540396\n|switch|p2a: Jolteon|Jolteon, L84, F|153/246\n|turn|20\n|\n|t:|1781540399\n|switch|p1a: Carbink|Carbink, L90|141/236\n|move|p2a: Jolteon|Substitute|p2a: Jolteon\n|-start|p2a: Jolteon|Substitute\n|-damage|p2a: Jolteon|92/246\n|\n|-heal|p2a: Jolteon|107/246|[from] item: Leftovers\n|upkeep\n|turn|21\n|\n|t:|1781540401\n|move|p2a: Jolteon|Calm Mind|p2a: Jolteon\n|-boost|p2a: Jolteon|spa|1\n|-boost|p2a: Jolteon|spd|1\n|move|p1a: Carbink|Body Press|p2a: Jolteon\n|-end|p2a: Jolteon|Substitute\n|\n|-heal|p2a: Jolteon|122/246|[from] item: Leftovers\n|upkeep\n|turn|22\n|\n|t:|1781540403\n|move|p2a: Jolteon|Thunderbolt|p1a: Carbink\n|-damage|p1a: Carbink|44/236\n|move|p1a: Carbink|Body Press|p2a: Jolteon\n|-damage|p2a: Jolteon|8/246\n|\n|-heal|p2a: Jolteon|23/246|[from] item: Leftovers\n|upkeep\n|turn|23\n|\n|t:|1781540406\n|move|p2a: Jolteon|Thunderbolt|p1a: Carbink\n|-damage|p1a: Carbink|0 fnt\n|faint|p1a: Carbink\n|\n|-heal|p2a: Jolteon|38/246|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540407\n|switch|p1a: Cyclizar|Cyclizar, L83, M|252/252\n|turn|24\n|\n|t:|1781540410\n|-terastallize|p2a: Jolteon|Ice\n|move|p2a: Jolteon|Tera Blast|p1a: Cyclizar|[anim] Tera Blast Ice\n|-supereffective|p1a: Cyclizar\n|-damage|p1a: Cyclizar|0 fnt\n|faint|p1a: Cyclizar\n|\n|-heal|p2a: Jolteon|53/246|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540416\n|switch|p1a: Mewtwo|Mewtwo, L72|272/272\n|-ability|p1a: Mewtwo|Unnerve\n|turn|25\n|\n|t:|1781540423\n|move|p2a: Jolteon|Tera Blast|p1a: Mewtwo|[anim] Tera Blast Ice\n|-damage|p1a: Mewtwo|127/272\n|move|p1a: Mewtwo|Psystrike|p2a: Jolteon\n|-damage|p2a: Jolteon|0 fnt\n|faint|p2a: Jolteon\n|-damage|p1a: Mewtwo|100/272|[from] item: Life Orb\n|\n|upkeep\n|\n|t:|1781540429\n|switch|p2a: Staraptor|Staraptor, L79, M|263/263\n|turn|26\n|\n|t:|1781540433\n|move|p1a: Mewtwo|Psystrike|p2a: Staraptor\n|-damage|p2a: Staraptor|68/263\n|-damage|p1a: Mewtwo|73/272|[from] item: Life Orb\n|move|p2a: Staraptor|U-turn|p1a: Mewtwo\n|-supereffective|p1a: Mewtwo\n|-damage|p1a: Mewtwo|0 fnt\n|faint|p1a: Mewtwo\n|\n|t:|1781540436\n|switch|p2a: Scrafty|Scrafty, L83, M|107/244 slp|[from] U-turn\n|\n|-heal|p2a: Scrafty|122/244 slp|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540440\n|switch|p1a: Ho-Oh|Ho-Oh, L71|260/268\n|turn|27\n|\n|t:|1781540448\n|move|p1a: Ho-Oh|Brave Bird|p2a: Scrafty\n|-supereffective|p2a: Scrafty\n|-damage|p2a: Scrafty|0 fnt\n|faint|p2a: Scrafty\n|-damage|p1a: Ho-Oh|220/268|[from] Recoil\n|\n|upkeep\n|\n|t:|1781540452\n|switch|p2a: Staraptor|Staraptor, L79, M|68/263\n|turn|28\n|\n|t:|1781540460\n|move|p2a: Staraptor|Brave Bird|p1a: Ho-Oh\n|-damage|p1a: Ho-Oh|0 fnt\n|faint|p1a: Ho-Oh\n|-damage|p2a: Staraptor|0 fnt|[from] Recoil\n|faint|p2a: Staraptor\n|\n|upkeep\n|\n|t:|1781540462\n|switch|p2a: Hitmonlee|Hitmonlee, L85, M|224/224\n|switch|p1a: Lugia|Lugia, L73|275/275\n|turn|29\n|\n|t:|1781540468\n|move|p1a: Lugia|Aeroblast|p2a: Hitmonlee\n|-supereffective|p2a: Hitmonlee\n|-damage|p2a: Hitmonlee|102/224\n|move|p2a: Hitmonlee|Knock Off|p1a: Lugia\n|-supereffective|p1a: Lugia\n|-damage|p1a: Lugia|208/275\n|-enditem|p1a: Lugia|Heavy-Duty Boots|[from] move: Knock Off|[of] p2a: Hitmonlee\n|\n|upkeep\n|turn|30\n|\n|t:|1781540475\n|-terastallize|p1a: Lugia|Ground\n|move|p1a: Lugia|Aeroblast|p2a: Hitmonlee\n|-supereffective|p2a: Hitmonlee\n|-damage|p2a: Hitmonlee|0 fnt\n|faint|p2a: Hitmonlee\n|\n|win|xValk_\n|raw|xValk_'s rating: 2138 &rarr; <strong>2158</strong><br />(+20 for winning)\n|raw|manwhocouldntcook's rating: 2146 &rarr; <strong>2126</strong><br />(-20 for losing)\n"

In [82]:
log2 = "|j|☆xxzfn\n|j|☆Bromeomeo\n|t:|1781540288\n|gametype|singles\n|player|p1|xxzfn|170|2221\n|player|p2|Bromeomeo|acetrainersnow|2186\n|gen|9\n|tier|[Gen 9] Random Battle\n|rated|\n|rule|Species Clause: Limit one of each Pokémon\n|rule|HP Percentage Mod: HP is shown in percentages\n|rule|Sleep Clause Mod: Limit one foe put to sleep\n|rule|Illusion Level Mod: Illusion disguises the Pokémon's true level\n|\n|t:|1781540288\n|teamsize|p1|6\n|teamsize|p2|6\n|start\n|switch|p1a: Ariados|Ariados, L95, F|287/287\n|switch|p2a: Weavile|Weavile, L79, M|239/239\n|turn|1\n|inactive|Battle timer is ON: inactive players will automatically lose when time's up. (requested by xxzfn)\n|\n|t:|1781540310\n|move|p2a: Weavile|Triple Axel|p1a: Ariados\n|-damage|p1a: Ariados|253/287\n|-damage|p1a: Ariados|187/287\n|-damage|p1a: Ariados|91/287\n|-hitcount|p1a: Ariados|3\n|move|p1a: Ariados|Sticky Web|p2a: Weavile\n|-sidestart|p2: Bromeomeo|move: Sticky Web\n|\n|upkeep\n|turn|2\n|\n|t:|1781540326\n|move|p2a: Weavile|Triple Axel|p1a: Ariados|[miss]\n|-miss|p2a: Weavile|p1a: Ariados\n|move|p1a: Ariados|Megahorn|p2a: Weavile|[miss]\n|-miss|p1a: Ariados|p2a: Weavile\n|\n|upkeep\n|turn|3\n|\n|t:|1781540332\n|move|p2a: Weavile|Triple Axel|p1a: Ariados\n|-damage|p1a: Ariados|54/287\n|-damage|p1a: Ariados|0 fnt\n|faint|p1a: Ariados\n|-hitcount|p1: Ariados|2\n|\n|upkeep\n|\n|t:|1781540335\n|switch|p1a: Clefable|Clefable, L83, M|293/293\n|turn|4\n|\n|t:|1781540340\n|-terastallize|p2a: Weavile|Ice\n|move|p2a: Weavile|Triple Axel|p1a: Clefable\n|-damage|p1a: Clefable|243/293\n|-damage|p1a: Clefable|143/293\n|-damage|p1a: Clefable|0 fnt\n|faint|p1a: Clefable\n|-hitcount|p1: Clefable|3\n|\n|upkeep\n|\n|t:|1781540350\n|switch|p1a: Lurantis|Lurantis, L87, F|264/264\n|turn|5\n|\n|t:|1781540359\n|switch|p2a: Darkrai|Darkrai, L77|234/234\n|-activate|p2a: Darkrai|move: Sticky Web\n|-unboost|p2a: Darkrai|spe|1\n|-terastallize|p1a: Lurantis|Fighting\n|move|p1a: Lurantis|Superpower|p2a: Darkrai\n|-supereffective|p2a: Darkrai\n|-damage|p2a: Darkrai|0 fnt\n|-boost|p1a: Lurantis|atk|1\n|-boost|p1a: Lurantis|def|1\n|faint|p2a: Darkrai\n|\n|upkeep\n|\n|t:|1781540364\n|switch|p2a: Dragalge|Dragalge, L88, M|258/258\n|-activate|p2a: Dragalge|move: Sticky Web\n|-unboost|p2a: Dragalge|spe|1\n|turn|6\n|\n|t:|1781540370\n|switch|p1a: Clodsire|Clodsire, L81, F|343/343\n|move|p2a: Dragalge|Sludge Wave|p1a: Clodsire\n|-resisted|p1a: Clodsire\n|-damage|p1a: Clodsire|311/343\n|\n|-heal|p1a: Clodsire|332/343|[from] item: Leftovers\n|upkeep\n|turn|7\n|\n|t:|1781540380\n|switch|p2a: Inteleon|Inteleon, L81, F|246/246\n|-activate|p2a: Inteleon|move: Sticky Web\n|-unboost|p2a: Inteleon|spe|1\n|move|p1a: Clodsire|Earthquake|p2a: Inteleon\n|-damage|p2a: Inteleon|138/246\n|\n|-heal|p1a: Clodsire|343/343|[from] item: Leftovers\n|upkeep\n|turn|8\n|\n|t:|1781540384\n|move|p2a: Inteleon|Scald|p1a: Clodsire\n|-immune|p1a: Clodsire|[from] ability: Water Absorb\n|move|p1a: Clodsire|Curse|p1a: Clodsire\n|-unboost|p1a: Clodsire|spe|1\n|-boost|p1a: Clodsire|atk|1\n|-boost|p1a: Clodsire|def|1\n|\n|upkeep\n|turn|9\n|\n|t:|1781540391\n|switch|p2a: Crabominable|Crabominable, L90, F|321/321\n|-activate|p2a: Crabominable|move: Sticky Web\n|-unboost|p2a: Crabominable|spe|1\n|move|p1a: Clodsire|Curse|p1a: Clodsire\n|-unboost|p1a: Clodsire|spe|1\n|-boost|p1a: Clodsire|atk|1\n|-boost|p1a: Clodsire|def|1\n|\n|upkeep\n|turn|10\n|\n|t:|1781540396\n|move|p2a: Crabominable|Ice Hammer|p1a: Clodsire\n|-supereffective|p1a: Clodsire\n|-damage|p1a: Clodsire|0 fnt\n|-unboost|p2a: Crabominable|spe|1\n|faint|p1a: Clodsire\n|\n|upkeep\n|\n|t:|1781540400\n|switch|p1a: Lurantis|Lurantis, L87, F, tera:Fighting|264/264\n|turn|11\n|\n|t:|1781540406\n|move|p1a: Lurantis|Superpower|p2a: Crabominable\n|-supereffective|p2a: Crabominable\n|-damage|p2a: Crabominable|43/321\n|-boost|p1a: Lurantis|atk|1\n|-boost|p1a: Lurantis|def|1\n|move|p2a: Crabominable|Ice Hammer|p1a: Lurantis|[miss]\n|-miss|p2a: Crabominable|p1a: Lurantis\n|\n|upkeep\n|turn|12\n|\n|t:|1781540411\n|move|p1a: Lurantis|Superpower|p2a: Crabominable\n|-supereffective|p2a: Crabominable\n|-damage|p2a: Crabominable|0 fnt\n|-boost|p1a: Lurantis|atk|1\n|-boost|p1a: Lurantis|def|1\n|faint|p2a: Crabominable\n|\n|upkeep\n|\n|t:|1781540414\n|switch|p2a: Skeledirge|Skeledirge, L79, M|294/294\n|turn|13\n|\n|t:|1781540418\n|move|p2a: Skeledirge|Torch Song|p1a: Lurantis\n|-damage|p1a: Lurantis|186/264\n|-boost|p2a: Skeledirge|spa|1\n|move|p1a: Lurantis|Knock Off|p2a: Skeledirge\n|-supereffective|p2a: Skeledirge\n|-damage|p2a: Skeledirge|150/294\n|-enditem|p2a: Skeledirge|Heavy-Duty Boots|[from] move: Knock Off|[of] p1a: Lurantis\n|\n|-heal|p1a: Lurantis|202/264|[from] item: Leftovers\n|upkeep\n|turn|14\n|\n|t:|1781540427\n|switch|p1a: Mismagius|Mismagius, L86, M|243/243\n|move|p2a: Skeledirge|Flame Charge|p1a: Mismagius\n|-damage|p1a: Mismagius|192/243\n|-boost|p2a: Skeledirge|spe|1\n|\n|-heal|p1a: Mismagius|207/243|[from] item: Leftovers\n|upkeep\n|turn|15\n|\n|t:|1781540432\n|switch|p2a: Dragalge|Dragalge, L88, M|258/258\n|-activate|p2a: Dragalge|move: Sticky Web\n|-unboost|p2a: Dragalge|spe|1\n|move|p1a: Mismagius|Shadow Ball|p2a: Dragalge\n|-damage|p2a: Dragalge|207/258\n|\n|-heal|p1a: Mismagius|222/243|[from] item: Leftovers\n|upkeep\n|turn|16\n|\n|t:|1781540438\n|move|p1a: Mismagius|Substitute|p1a: Mismagius\n|-start|p1a: Mismagius|Substitute\n|-damage|p1a: Mismagius|162/243\n|move|p2a: Dragalge|Dragon Tail|p1a: Mismagius\n|-end|p1a: Mismagius|Substitute\n|\n|-heal|p1a: Mismagius|177/243|[from] item: Leftovers\n|upkeep\n|turn|17\n|\n|t:|1781540445\n|move|p1a: Mismagius|Shadow Ball|p2a: Dragalge\n|-damage|p2a: Dragalge|159/258\n|move|p2a: Dragalge|Dragon Tail|p1a: Mismagius|[miss]\n|-miss|p2a: Dragalge|p1a: Mismagius\n|\n|-heal|p1a: Mismagius|192/243|[from] item: Leftovers\n|upkeep\n|turn|18\n|\n|t:|1781540448\n|move|p1a: Mismagius|Shadow Ball|p2a: Dragalge\n|-damage|p2a: Dragalge|111/258\n|move|p2a: Dragalge|Dragon Tail|p1a: Mismagius|[miss]\n|-miss|p2a: Dragalge|p1a: Mismagius\n|\n|-heal|p1a: Mismagius|207/243|[from] item: Leftovers\n|upkeep\n|turn|19\n|\n|t:|1781540451\n|move|p1a: Mismagius|Shadow Ball|p2a: Dragalge\n|-damage|p2a: Dragalge|68/258\n|-unboost|p2a: Dragalge|spd|1\n|move|p2a: Dragalge|Dragon Tail|p1a: Mismagius\n|-damage|p1a: Mismagius|107/243\n|drag|p1a: Hitmonlee|Hitmonlee, L85, M|223/223\n|\n|upkeep\n|turn|20\n|\n|t:|1781540454\n|move|p1a: Hitmonlee|Knock Off|p2a: Dragalge\n|-damage|p2a: Dragalge|0 fnt\n|-enditem|p2a: Dragalge|Assault Vest|[from] move: Knock Off|[of] p1a: Hitmonlee\n|faint|p2a: Dragalge\n|\n|upkeep\n|\n|t:|1781540459\n|switch|p2a: Inteleon|Inteleon, L81, F|138/246\n|-activate|p2a: Inteleon|move: Sticky Web\n|-unboost|p2a: Inteleon|spe|1\n|turn|21\n|\n|t:|1781540464\n|move|p1a: Hitmonlee|Knock Off|p2a: Inteleon\n|-damage|p2a: Inteleon|0 fnt\n|-enditem|p2a: Inteleon|Choice Specs|[from] move: Knock Off|[of] p1a: Hitmonlee\n|faint|p2a: Inteleon\n|\n|upkeep\n|\n|t:|1781540466\n|switch|p2a: Weavile|Weavile, L79, M, tera:Ice|239/239\n|-activate|p2a: Weavile|move: Sticky Web\n|-unboost|p2a: Weavile|spe|1\n|turn|22\n|\n|t:|1781540472\n|move|p1a: Hitmonlee|Knock Off|p2a: Weavile\n|-damage|p2a: Weavile|79/239\n|-enditem|p2a: Weavile|Choice Band|[from] move: Knock Off|[of] p1a: Hitmonlee\n|-enditem|p1a: Hitmonlee|Choice Band|[silent]|[from] ability: Pickpocket|[of] p1a: Hitmonlee\n|-item|p2a: Weavile|Choice Band|[from] ability: Pickpocket|[of] p1a: Hitmonlee\n|move|p2a: Weavile|Triple Axel|p1a: Hitmonlee\n|-damage|p1a: Hitmonlee|165/223\n|-damage|p1a: Hitmonlee|49/223\n|-damage|p1a: Hitmonlee|0 fnt\n|faint|p1a: Hitmonlee\n|-hitcount|p1: Hitmonlee|3\n|\n|upkeep\n|\n|t:|1781540480\n|switch|p1a: Mismagius|Mismagius, L86, M|107/243\n|turn|23\n|\n|t:|1781540486\n|switch|p2a: Skeledirge|Skeledirge, L79, M|150/294\n|-activate|p2a: Skeledirge|move: Sticky Web\n|-unboost|p2a: Skeledirge|spe|1\n|move|p1a: Mismagius|Shadow Ball|p2a: Skeledirge\n|-supereffective|p2a: Skeledirge\n|-damage|p2a: Skeledirge|0 fnt\n|faint|p2a: Skeledirge\n|\n|-heal|p1a: Mismagius|122/243|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540487\n|switch|p2a: Weavile|Weavile, L79, M, tera:Ice|79/239\n|-activate|p2a: Weavile|move: Sticky Web\n|-unboost|p2a: Weavile|spe|1\n|turn|24\n|\n|t:|1781540492\n|move|p2a: Weavile|Ice Shard|p1a: Mismagius\n|-crit|p1a: Mismagius\n|-damage|p1a: Mismagius|0 fnt\n|faint|p1a: Mismagius\n|\n|upkeep\n|\n|t:|1781540494\n|switch|p1a: Lurantis|Lurantis, L87, F, tera:Fighting|202/264\n|turn|25\n|\n|t:|1781540496\n|move|p2a: Weavile|Ice Shard|p1a: Lurantis\n|-damage|p1a: Lurantis|116/264\n|move|p1a: Lurantis|Superpower|p2a: Weavile\n|-supereffective|p2a: Weavile\n|-damage|p2a: Weavile|0 fnt\n|-boost|p1a: Lurantis|atk|1\n|-boost|p1a: Lurantis|def|1\n|faint|p2a: Weavile\n|\n|win|xxzfn\n"

In [83]:
battle1 = Battle(log1)
battle1

This was a battle between xValk_ (pre-match rating 2138) and manwhocouldntcook (pre-match rating 2146).
xValk_'s team was {'Carbink', 'Ho-Oh', 'Lugia', 'Victreebel', 'Cyclizar', 'Mewtwo'}.
manwhocouldntcook's team was {'Incineroar', 'Jolteon', 'Hitmonlee', 'Vileplume', 'Staraptor', 'Scrafty'}.
xValk_ won!
xValk_'s rating increased to 2158.
manwhocouldntcook's rating fell to 2126.

In [84]:
battle2 = Battle(log2)
battle2

This was a battle between xxzfn (pre-match rating 2221) and Bromeomeo (pre-match rating 2186).
xxzfn's team was {'Ariados', 'Mismagius', 'Lurantis', 'Clefable', 'Clodsire'}.
Bromeomeo's team was {'Crabominable', 'Dragalge', 'Inteleon', 'Skeledirge', 'Darkrai', 'Weavile'}.
xxzfn won!
This was an unrated match, so no one's rating changed.

# Sandbox

Below is what I was using to test.  You should probably ignore this stuff unless you want to run some test of your own without needing to type "self" everywhere...

In [58]:
player_names = ["",""]
old_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings at the start of the match
new_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings after the end of the match
winner = ""
loser = ""
teams = {} # a dictionary where the keys are player usernames and the values are sets containing the pokemon names
time_list = [] # for keeping track of all move times, gets thrown away now.  Could be used to check for time-out forfeits?
start_time = 0
end_time = 0

for i in range(len(log)):
    # generates player usernames, initializes the teams dict with an empty set for each player, sets the old_player_ratings
    if log.startswith("|player|",i):
        j = i+11
        # grabs the index of the player: 1 if p2, 0 else; this way, we avoid the assumption that player 1's name is printed first
        player_index = int(log.startswith("|p2|",i+7))
        while log[j] != "|":
            player_names[player_index] += log[j]
            j += 1
        # initializes the teams as empty sets
        teams[player_names[player_index]] = set()
        # grabs the initial ratings of each player; tricky because you want to grab the last 3 or 4 characters of the line depending on 
        # whether the player's rating is 3 or 4 digits
        while log[j] != "\n":
            j += 1
        # j is now the index of the \n character at the end of the line
        k = 0
        while log[j - k] != "|":
            k = k+1
        # j - k is now the index of the | before the player's rating
        old_player_ratings[player_names[player_index]] = int(log[j-k+1:j])
    # generates time list
    if log.startswith("|t:",i):
        j = i + 4
        time_string = ""
        while log[j] != "\n":
            time_string += log[j]
            j += 1
        time_list.append(int(time_string))
    # generate team list
    if log.startswith("|switch|",i):
        pokemon_name = ""
        j = i+13
        while log[j] != "|":
            pokemon_name += log[j]
            j += 1
        if log.startswith("p1a",i+8):
            player_name = player_names[0]
        else:
            player_name = player_names[1]
        teams[player_name].add(pokemon_name)
    # identifies winner and loser
    if log.startswith("|win|",i):
        j = i + 5
        while log[j] != "\n":
            winner += log[j]
            j += 1
        winner_index = player_names.index(winner)
        loser = player_names[1-winner_index]
    if log.startswith(" for winning)",i):
        delta_rating = int(log[i-2:i]) # assumes that the change in rating will be two digits
        new_player_ratings[winner] = old_player_ratings[winner] + delta_rating
        new_player_ratings[loser] = old_player_ratings[loser] - delta_rating
    

start_time = time_list[0]
end_time = time_list[-1]

for index in range(2):
    print(f"{player_names[index]}'s old rating:", old_player_ratings[player_names[index]])
    print(f"{player_names[index]}'s team:", teams[player_names[index]])
print("winner:", winner)
try:
    for index in range(2):
        print(f"{player_names[index]} new rating:", new_player_ratings[player_names[index]])
except KeyError:
    print("This was not a rated match")

print("start time:", start_time)
print("end time:", end_time)

xValk_'s old rating: 2138
xValk_'s team: {'Carbink', 'Ho-Oh', 'Lugia', 'Victreebel', 'Cyclizar', 'Mewtwo'}
manwhocouldntcook's old rating: 2146
manwhocouldntcook's team: {'Incineroar', 'Jolteon', 'Hitmonlee', 'Vileplume', 'Staraptor', 'Scrafty'}
winner: xValk_
xValk_ new rating: 2158
manwhocouldntcook new rating: 2126
start time: 1781540286
end time: 1781540475


In [85]:
line = "|player|p1|xxzfn|170|2221\n"
j = 11
# grabs the index of the player: 1 if p2, 0 else; this way, we avoid the assumption that player 1's name is printed first
while line[j] != "|":
    j += 1
# grabs the initial ratings of each player; tricky because you want to grab the last 3 or 4 characters of the line depending on 
# whether the player's rating is 3 or 4 digits
while line[j] != "\n":
    j += 1
    print(f"line[{j}] = {line[j]}")
# j is now the index of the \n character at the end of the line
k = 0
while line[j - k] != "|":
    k = k+1
    print(f"k = {k}")
# j - k is now the index of the | before the player's rating
# old_player_ratings[player_names[player_index]] = int(log[j-k+1:j-1])
print("rating:",line[j-k+1:j])


line[17] = 1
line[18] = 7
line[19] = 0
line[20] = |
line[21] = 2
line[22] = 2
line[23] = 2
line[24] = 1
line[25] = 

k = 1
k = 2
k = 3
k = 4
k = 5
rating: 2221


In [43]:
len(line)

26